In [1]:
import pandas as pd
import numpy as np

In [2]:
!curl -o car_fuel_efficiency_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100 620.2k 100 620.2k   0      0 867.9k      0                              0
100 620.2k 100 620.2k   0      0 867.8k      0                              0
100 620.2k 100 620.2k   0      0 867.7k      0                              0


In [3]:
df=pd.read_csv("car_fuel_efficiency_2026.csv")

In [4]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [5]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [7]:
df["horsepower"].median()

np.float64(254.0)

In [9]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test
cols = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]
df = df[cols]
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]


In [10]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

feature_cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    return w[0], w[1:]

def rmse(y, y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

X_train_0 = df_train[feature_cols].fillna(0).values
w0_0, w_0 = train_linear_regression(X_train_0, y_train)

X_val_0 = df_val[feature_cols].fillna(0).values
y_pred_0 = w0_0 + X_val_0 @ w_0
score_0 = round(rmse(y_val, y_pred_0), 3)

mean_hp = df_train['horsepower'].mean()

X_train_mean = df_train[feature_cols].fillna({'horsepower': mean_hp}).values
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

X_val_mean = df_val[feature_cols].fillna({'horsepower': mean_hp}).values
y_pred_mean = w0_mean + X_val_mean @ w_mean
score_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"RMSE (With 0):    {score_0}")
print(f"RMSE (With mean): {score_mean}")

RMSE (With 0):    2.205
RMSE (With mean): 2.202


In [11]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y

    return w[0], w[1:]

X_train = df_train[feature_cols].fillna(0).values
X_val = df_val[feature_cols].fillna(0).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val @ w
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:5} -> RMSE: {score}")

r =     0 -> RMSE: 2.2053
r =  0.01 -> RMSE: 2.2058
r =   0.1 -> RMSE: 2.2241
r =     1 -> RMSE: 2.3492
r =     5 -> RMSE: 2.4094
r =    10 -> RMSE: 2.4195
r =   100 -> RMSE: 2.4292


In [12]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    X_train = df_train[feature_cols].fillna(0).values
    X_val = df_val[feature_cols].fillna(0).values

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val @ w
    scores.append(rmse(y_val, y_pred))

std_score = round(np.std(scores), 3)
print(f"std: {std_score}")

std: 0.029
